# Simple Active Learning

Basic tutorial on `SimpleActiveLearningJourney`.

In [ ]:
from scm.moliterate import PropertyInfo
from scm.moliterate.analysis import PairwiseDatasetMetrics
from scm.plams import from_smiles

from scm.active_learning import ActiveLearningLoop, SimpleActiveLearningJourney
from scm.active_learning.checker_getters.checkers import AMSTrajChecker
from scm.active_learning.checker_getters.getters import AMSTrajGetter
from scm.active_learning.engines import AMSEngine, SCMChemicalSystem
from scm.active_learning.loop import StoppableCounter
from scm.active_learning.mlip.params_trainer import ParAMSTrainer
from scm.active_learning.tasks import AMSMDTask, SPLabeller

In [ ]:
sal = SimpleActiveLearningJourney(
    checker_getter=AMSTrajChecker() + AMSTrajGetter(),
    task=AMSMDTask(
        samplingfreq=2,
        thermostat="NHC",
        atomistic_system=SCMChemicalSystem.from_molecules(
            system_id="CO", 
            systems=from_smiles("CO"),
        ),
    ),
    steps = SimpleActiveLearningJourney.ListSteps(cumulative_values=[50, 120]),
    max_attempts=3,
    first_step_train=True,
)

In [2]:
print(sal.journey_table())

=================================== JourneyScheduler ===================================
AMSMDTask|CO -> AMSTrajChecker|AMSTrajGetter
  MD NSteps    MD NSteps - Cumulative    MD SamplingFreq    MD MaxFrames    max_attempts
-----------  ------------------------  -----------------  --------------  --------------
         50                        50                  1              50               3
         70                       120                  1              70               3


In [3]:
import sys

ActiveLearningLoop.logging_config(
    clean_sinks="YES",
    new_sink=sys.stderr,
    level="INFO",
)

1

In [4]:
from scm.active_learning.task_parallelization import AMSParallelCPU

al = ActiveLearningLoop(
    start_engine=AMSEngine.Builder.UFF().build(),
    journey=sal,
    iterable_loop=StoppableCounter(stop=5),
    labeller=SPLabeller(
        properties=[
            PropertyInfo(name="energy", unit="eV"),
            PropertyInfo(name="forces", unit="eV/Ang"),
        ],
        parallelization=AMSParallelCPU(maxjobs=5, nproc=1),
    ),
    labeller_engine=AMSEngine.Builder.UFF().build(),
    # labeller_engine=AMSEngine.Builder.DFTB_GFN1().build(),
    accuracy_checker = PairwiseDatasetMetrics(settings=[
        PairwiseDatasetMetrics.PropMetricEv(property="energy", metric="mae", per_n_atoms=True, target=0.05)
    ]),
    # mlip_trainer=ParAMSTrainer.Builder.M3GNet_EF().set_max_epochs(5).set_committee(1).build(),
    mlip_trainer=ParAMSTrainer.Builder.TEST().set_committee(1).build(),
)

In [5]:
al.run()

2026-04-21 15:27:00 | INFO     | AL Loop Folder: ALruns/20260421_152700
2026-04-21 15:27:00 | INFO     | Train Val created: ALruns/20260421_152700/train_validation_AL.db
2026-04-21 15:27:00 | INFO     | 
=================================== JourneyScheduler ===================================
AMSMDTask|CO -> AMSTrajChecker|AMSTrajGetter
  MD NSteps    MD NSteps - Cumulative    MD SamplingFreq    MD MaxFrames    max_attempts
-----------  ------------------------  -----------------  --------------  --------------
         50                        50                  1              50               3
         70                       120                  1              70               3
2026-04-21 15:27:00 | INFO     | ManualStopper: to stop the loop manually, run:
 echo "message" > /home/bene/Documents/work/ALIR/Code/active_learning_workspace/active_learning/tutorials/notebooks/ALruns/20260421_152700/STOP_ACTIVE_LEARNING
2026-04-21 15:27:00 | INFO     | ===================== iAL:00 ====

ParAMSEngine(engine_id='MLIPTest01', type='ParAMSEngine', source_settings=[{'input': {'ForceField': {'Type': 'uff'}}}], finetunable=True, job_path='/home/bene/Documents/work/ALIR/Code/active_learning_workspace/active_learning/tutorials/notebooks/ALruns/20260421_152700/iter_01/ParAMSTrainer', nproc=1, add_OMP_NUM_THREADS=1)

In [6]:
al.analysis.get_summary()

{'Reason': 'AL stopped: CONVERGED',
 'NSteps': 3,
 'Ninit': 0,
 'Nfinal': 2,
 'Tot[min]': 0}

In [7]:
import webbrowser

webbrowser.open(str(al.analysis.plot.save_to_pdf()))

ALruns/20260421_152700/analysis.pdf


True

In [8]:
print(al.journey.history_table())

=============================================== JourneyHistory ===============================================
  Idx  type        step    n_steps_left    attempt  finished_msg                          finished    success
-----  --------  ------  --------------  ---------  ------------------------------------  ----------  ---------
    0  SALState       0               1          1  Validity AND Accuracy checks passed.  True        True
    1  SALState       1               0          1  Validity AND Accuracy checks passed.  True        True


In [9]:
import shutil

from scm.active_learning.callbacks import (
    FolderManagerCallback,
)

shutil.rmtree(al.query_callbacks(FolderManagerCallback)[0].run_dir())